# Workflow demo: parallel generatsiya + evaluator-optimizer

Bu notebook **agent emas, workflow**: qadamlar kodda oldindan yozilgan, model faqat matn yozadi va baholaydi.

**Vazifa:** uzun matnning har bir bo'limiga test savollari tuzish, ularni tekshirish va yomonlarini qayta tuzish.

```
5 bo'lim ──► [PARALLEL: har bo'limga 2 tadan test] ──► [PARALLEL: evaluator har testni tekshiradi]
                                                            │
                                                  ok=False  ▼
                                              [OPTIMIZER: sabab bilan qayta tuzish, 1 marta]
                                                            │
                                                            ▼
                                                   [qayta tekshirish] ──► natija
```

**Bu yerda ko'radiganlaringiz:**
1. `async` / `await` nima va nega kerak (API'siz, oddiy misolda)
2. `asyncio.gather` — bir nechta LLM chaqiruvini bir vaqtda yuborish
3. `Semaphore` — bir vaqtdagi so'rovlar sonini cheklash (rate limit'dan saqlanish)
4. Pydantic + structured output — test va hukmni obyekt qilib olish
5. Evaluator — boshqa LLM chaqiruvi testni tekshiradi
6. Optimizer — xato sababi bilan qayta tuzish
7. Ketma-ket va parallel vaqtini solishtirish

Hammasi bitta uzun matn bilan ishlaydi (Ipak yo'li haqida), fayl o'qilmaydi.

## 0. Sozlamalar

`AsyncOpenAI` — oddiy `OpenAI` ning asinxron versiyasi. Metodlari bir xil (`responses.parse`), faqat oldiga `await` yoziladi.

In [2]:
import asyncio, time, json
from typing import Literal
from pydantic import BaseModel, Field
from openai import AsyncOpenAI

client = AsyncOpenAI()
MODEL = "gpt-6-luna"

## 1. Matn

Faylsiz ishlash uchun matn shu yerning o'zida. Bo'limlar `## ` sarlavha bilan ajratilgan — keyingi katakchada ularni ro'yxatga bo'lamiz. Bu hujjatdagi "chunk"larning o'rniga.

In [1]:
TEXT = '''
## Ipak yo'lining paydo bo'lishi
Ipak yo'li — Xitoyni O'rta Osiyo, Eron, Yaqin Sharq va O'rta yer dengizi bilan bog'lagan savdo yo'llari tarmog'i. Uning boshlanishi miloddan avvalgi II asrga, Xitoydagi Xan sulolasi davriga borib taqaladi. Xan imperatori elchi Chjan Szyanni g'arbga yuborgan va uning sayohatlari natijasida Xitoy g'arbdagi xalqlar, ayniqsa Farg'ona vodiysining kuchli otlari haqida ma'lumot olgan. "Ipak yo'li" atamasi esa ancha keyin, XIX asrda nemis geografi Ferdinand fon Rixtgofen tomonidan qo'llanilgan. Bu yo'l bitta yo'l emas, balki karvonlar o'tadigan ko'plab yo'nalishlardan iborat bo'lgan.

## Savdo qilingan mollar
Yo'lning nomi Xitoy ipagidan olingan: ipak G'arbda juda qimmat baholangan va uzoq vaqt uni ishlab chiqarish siri faqat Xitoyda saqlangan. Sharqdan g'arbga ipak, chinni, choy va ziravorlar olib borilgan. G'arbdan sharqqa esa oltin, kumush, shisha buyumlar, jun mato va otlar keltirilgan. Savdo odatda bosqichma-bosqich bo'lgan: bitta savdogar butun yo'lni bosib o'tmagan, mollar shaharma-shahar qo'ldan qo'lga o'tib borgan. Shu sababli mol yo'lning oxiriga yetganda narxi bir necha barobar oshib ketgan.

## O'rta Osiyo shaharlari
Ipak yo'lida O'rta Osiyo shaharlari muhim o'rin tutgan. Samarqand, Buxoro, Marv va Xiva karvonlar to'xtaydigan, mol almashinadigan va dam oladigan markazlar bo'lgan. Karvonlar uchun yo'l bo'ylab karvonsaroylar qurilgan: ular mehmonxona, ombor va bozor vazifasini bajargan. Sug'diylar — Samarqand atrofida yashagan xalq — Ipak yo'lidagi eng faol savdogarlardan bo'lgan, ularning tili uzoq vaqt savdo tili sifatida ishlatilgan. Shaharlar savdodan boyib, me'morchilik va hunarmandchilik rivojlangan.

## G'oyalar va bilimlar almashinuvi
Ipak yo'li orqali faqat mollar emas, g'oyalar ham tarqalgan. Buddizm Hindistondan O'rta Osiyo orqali Xitoyga yetib borgan, keyinroq islom ham shu yo'llar bo'ylab sharqqa tarqalgan. Qog'oz ishlab chiqarish texnologiyasi Xitoydan g'arbga o'tgan va VIII asrda Samarqandda qog'oz ishlab chiqarila boshlangan; Samarqand qog'ozi o'z davrida mashhur bo'lgan. Bundan tashqari astronomiya, matematika va tibbiyotga oid bilimlar ham turli xalqlar orasida almashilgan.

## Ipak yo'lining zaiflashuvi
XIII asrda Mo'g'ullar imperiyasi Yevrosiyoning katta qismini birlashtirgach, yo'llar nisbatan xavfsiz bo'lib qolgan va venetsiyalik sayyoh Marko Polo shu davrda Xitoygacha borgan. Biroq XV-XVI asrlarda yevropaliklar dengiz yo'llarini ochgach, mollarni kemada tashish arzonroq va tezroq bo'lib qolgan. Quruqlikdagi savdo asta-sekin kamaygan, siyosiy parchalanish va urushlar ham yo'llarni xavfli qilgan. Natijada Ipak yo'lining ahamiyati pasaygan, ammo uning madaniy merosi bugungacha saqlanib qolgan.
'''

SECTIONS = []
for block in TEXT.strip().split("## ")[1:]:
    title, body = block.split("\n", 1)
    SECTIONS.append({"title": title.strip(), "text": body.strip()})

for s in SECTIONS:
    print(f"- {s['title']}: {len(s['text'].split())} so'z")

- Ipak yo'lining paydo bo'lishi: 79 so'z
- Savdo qilingan mollar: 69 so'z
- O'rta Osiyo shaharlari: 63 so'z
- G'oyalar va bilimlar almashinuvi: 59 so'z
- Ipak yo'lining zaiflashuvi: 61 so'z


## 2. Sxemalar (Pydantic)

Uchta sxema:
- `Question` — bitta test. **Diqqat:** `explanation` maydoni `correct_index` dan **oldin** turibdi — model avval asoslab, keyin javobni tanlaydi.
- `Quiz` — bitta bo'lim uchun testlar ro'yxati.
- `Verdict` — evaluator hukmi: `ok` va `reason`. `reason` optimizer uchun kerak: model nima xato qilganini biladi.

In [5]:
class Question(BaseModel):
    question: str = Field(description="Savol matni, o'zbek tilida")
    options: list[str] = Field(description="Aynan 4 ta javob varianti")
    explanation: str = Field(description="To'g'ri javob nega to'g'ri, matnga tayanib, 1-2 gap")
    correct_index: int = Field(description="To'g'ri variant indeksi, 0 dan 3 gacha")

class Quiz(BaseModel):
    questions: list[Question]

class Verdict(BaseModel):
    ok: bool = Field(description="Test sifatli va to'g'rimi")
    reason: str = Field(description="ok=false bo'lsa: aniq nima xato va qanday tuzatish kerak")

## 3. Async nima? (API'siz misol)

**Muammo:** LLM chaqiruvi ~5-10 soniya. Bu vaqtning deyarli hammasida sizning kompyuteringiz hech narsa qilmaydi — faqat **javobni kutadi**.

**Oshxona misoli:** 3 ta choynak qo'yish kerak, har biri 1 daqiqada qaynaydi.
- **Ketma-ket:** birinchisini qo'yasiz, qaynaguncha kutasiz, keyin ikkinchisini... → 3 daqiqa.
- **Parallel (async):** uchalasini birdan qo'yasiz, keyin kutasiz → 1 daqiqa.

Python'da:
- `async def` — "bu funksiya ichida kutish bor" degani.
- `await` — "shu yerda kut, lekin kutayotganda boshqa ishlarni qilib tur".
- `asyncio.gather(a, b, c)` — "uchalasini birdan boshla, hammasi tugaguncha kut".

Pastdagi misolda `asyncio.sleep(1)` LLM chaqiruvini taqlid qiladi.

**Jupyter'da:** katakchada to'g'ridan-to'g'ri `await` yozsa bo'ladi. Oddiy `.py` faylda esa `asyncio.run(main())` kerak bo'ladi.

In [3]:
async def fake_llm_call(name: str) -> str:
    await asyncio.sleep(1)          # 1 soniya "kutish" (tarmoq)
    return f"{name} tayyor"

# Ketma-ket
t0 = time.perf_counter()
for n in ["A", "B", "C"]:
    await fake_llm_call(n)
print(f"Ketma-ket: {time.perf_counter() - t0:.1f} s")

# Parallel
t0 = time.perf_counter()
results = await asyncio.gather(fake_llm_call("A"), fake_llm_call("B"), fake_llm_call("C"))
print(f"Parallel:  {time.perf_counter() - t0:.1f} s", results)

Ketma-ket: 3.0 s
Parallel:  1.0 s ['A tayyor', 'B tayyor', 'C tayyor']


## 4. Generator: bitta bo'limga test tuzish

**Semaphore** — bir vaqtda nechta so'rov ketishini cheklaydi. 1000 ta bo'limni birdaniga yuborsangiz, API **rate limit** xatosini beradi (production'dagi LLM xatolarining ~60% shu). `Semaphore(5)` — "bir vaqtda ko'pi bilan 5 ta".

`feedback` parametri optimizer uchun: avvalgi test nima uchun yomon bo'lganini modelga aytamiz.

In [6]:
SEM = asyncio.Semaphore(5)

async def generate_quiz(section: dict, n: int = 2, feedback: str | None = None) -> Quiz:
    rules = (f"Sen o'qituvchisan. Faqat BO'LIM matni asosida {n} ta test tuz. "
             "Har savolda aynan 4 ta variant, faqat bittasi to'g'ri, variantlar takrorlanmasin. "
             "Savol matnda aniq javobi bor narsa haqida bo'lsin.")
    if feedback:
        rules += f"\n\nOLDINGI URINISHDAGI XATO (buni takrorlama): {feedback}"
    async with SEM:
        resp = await client.responses.parse(
            model=MODEL,
            instructions=rules,
            input=f"BO'LIM: {section['title']}\n\n{section['text']}",
            text_format=Quiz,
        )
    return resp.output_parsed

## 5. Ketma-ket va parallel: vaqtni solishtirish

Ikkala usul ham aynan bir xil ishni qiladi — 5 bo'lim, har biriga 2 tadan test. Farq faqat **qanday** yuborilishida.

Natijani ko'rishdan oldin o'zingizdan so'rang: necha barobar tezroq bo'ladi?

In [7]:
t0 = time.perf_counter()
seq_quizzes = []
for s in SECTIONS:
    seq_quizzes.append(await generate_quiz(s))
t_seq = time.perf_counter() - t0

t0 = time.perf_counter()
quizzes = await asyncio.gather(*[generate_quiz(s) for s in SECTIONS])
t_par = time.perf_counter() - t0

print(f"Ketma-ket: {t_seq:.1f} s")
print(f"Parallel:  {t_par:.1f} s  →  {t_seq / t_par:.1f} barobar tez")

Ketma-ket: 22.1 s
Parallel:  4.3 s  →  5.2 barobar tez


In [8]:
for s, quiz in zip(SECTIONS, quizzes):
    print(f"\n### {s['title']}")
    for q in quiz.questions:
        print(" ?", q.question)
        for i, o in enumerate(q.options):
            print("   ", "✓" if i == q.correct_index else " ", o)


### Ipak yo'lining paydo bo'lishi
 ? Ipak yo‘lining boshlanishi qaysi davrga borib taqaladi?
    ✓ Miloddan avvalgi II asr, Xan sulolasi davriga
      Miloddan avvalgi I asr, Xan sulolasi davriga
      Milodiy II asr, Xan sulolasi davriga
      Milodiy XIX asr, Xan sulolasi davriga
 ? “Ipak yo‘li” atamasini XIX asrda kim qo‘llagan?
      Elchi Chjan Szyan
      Xan imperatori
    ✓ Ferdinand fon Rixtgofen
      Xitoydagi karvonboshi

### Savdo qilingan mollar
 ? Sharqdan g'arbga qanday mahsulotlar olib borilgan?
      Oltin, kumush va otlar
    ✓ Ipak, chinni, choy va ziravorlar
      Shisha buyumlar va jun mato
      Oltin, ipak va shisha buyumlar
 ? Savdo yo'lida mollar nima sababdan shaharma-shahar qo'ldan qo'lga o'tgan?
      Ipak ishlab chiqarish siri faqat Xitoyda saqlangani uchun
      Savdogarlar faqat otlarda yurgani uchun
    ✓ Bitta savdogar butun yo'lni bosib o'tmagani uchun
      Mollar faqat shaharlarda sotilgani uchun

### O'rta Osiyo shaharlari
 ? Ipak yo‘lida karvonla

## 6. Evaluator: testni boshqa LLM chaqiruvi tekshiradi

**Nega alohida chaqiruv:** test tuzgan model o'z ishini o'zi "yaxshi" deb baholashga moyil. Alohida, **qattiq** ko'rsatmali tekshiruvchi xatolarni ko'proq topadi (Anthropic tadqiqotidagi asosiy saboq).

Evaluator testni **asl matn** bilan solishtiradi — test tuzuvchining fikrini emas.

**Ikki qavat tekshiruv:**
1. **Kod** — qat'iy narsalar: variantlar soni 4 tami, indeks 0-3 orasidami, takror bormi. Bu tekin va 100% aniq.
2. **LLM** — ma'no: to'g'ri javob haqiqatan to'g'rimi, savol matndan chiqadimi.

Kodda tekshirib bo'ladigan narsani LLM'ga bermaymiz (kechagi qoida: qat'iy qoidalar — kodda).

In [9]:
def code_check(q: Question) -> str | None:
    if len(q.options) != 4:
        return f"{len(q.options)} ta variant, aynan 4 ta bo'lishi kerak"
    if not 0 <= q.correct_index < 4:
        return f"correct_index={q.correct_index}, 0-3 bo'lishi kerak"
    if len({o.strip().lower() for o in q.options}) != 4:
        return "variantlar takrorlangan"
    return None

async def evaluate(section: dict, q: Question) -> Verdict:
    if err := code_check(q):                       # 1-qavat: kod
        return Verdict(ok=False, reason=err)
    async with SEM:                                # 2-qavat: LLM
        resp = await client.responses.parse(
            model=MODEL,
            instructions=("Sen qattiq tekshiruvchisan. Testni faqat MATN asosida tekshir: "
                          "1) belgilangan to'g'ri javob matnga ko'ra haqiqatan to'g'rimi; "
                          "2) boshqa variantlardan biri ham to'g'ri bo'lib qolmaganmi; "
                          "3) savolning javobi matnda bormi. Shubha bo'lsa ok=false."),
            input=(f"MATN:\n{section['text']}\n\nSAVOL: {q.question}\n"
                   + "\n".join(f"{i}) {o}" for i, o in enumerate(q.options))
                   + f"\nBELGILANGAN TO'G'RI JAVOB: {q.correct_index}"),
            text_format=Verdict,
        )
    return resp.output_parsed

### Evaluator ishlayaptimi? Ataylab yomon test bilan sinash

Evaluator hech qachon xato topmasa, bu ikki narsani anglatishi mumkin: yoki testlar a'lo, yoki evaluator ishlamayapti. Buni bilish uchun unga **ataylab buzilgan** testlar beramiz. Bu evaluator'ning o'zini tekshirish.

In [10]:
bad_tests = [
    Question(question="Ipak yo'li atamasini kim qo'llagan?",
             options=["Marko Polo", "Chjan Szyan", "Ferdinand fon Rixtgofen", "Xan imperatori"],
             explanation="...", correct_index=0),            # noto'g'ri belgilangan
    Question(question="Samarqandda qog'oz qachon ishlab chiqarila boshlangan?",
             options=["VIII asrda", "VIII asrda", "XIII asrda", "II asrda"],
             explanation="...", correct_index=0),            # takroriy variant
]
for q in bad_tests:
    v = await evaluate(SECTIONS[0] if "atama" in q.question else SECTIONS[3], q)
    print("ok =", v.ok, "|", v.reason)

ok = False | Matnga ko‘ra, “Ipak yo‘li” atamasini XIX asrda Ferdinand fon Rixtgofen qo‘llagan. Belgilangan 0-variant — Marko Polo — noto‘g‘ri. To‘g‘ri javob: 2.
ok = False | variantlar takrorlangan


## 7. Hamma testlarni parallel tekshirish

In [11]:
pairs = [(s, q) for s, quiz in zip(SECTIONS, quizzes) for q in quiz.questions]
verdicts = await asyncio.gather(*[evaluate(s, q) for s, q in pairs])

bad = [(s, q, v) for (s, q), v in zip(pairs, verdicts) if not v.ok]
print(f"{len(pairs)} ta testdan {len(bad)} tasi yomon")
for s, q, v in bad:
    print(f"\n✗ [{s['title']}] {q.question}\n  sabab: {v.reason}")

10 ta testdan 0 tasi yomon


## 8. Optimizer: yomonlarini sababi bilan qayta tuzish

Yomon test bo'lgan bo'limni **evaluator aytgan sabab** bilan qayta tuzamiz va yangisini yana tekshiramiz.

**Faqat 1 marta.** Cheksiz "tuz → tekshir → tuz" sikli pul yeydi va ba'zan hech qachon tugamaydi (agentdagi `max_steps` bilan bir xil g'oya). 1 martadan keyin ham yomon bo'lsa — testni tashlab yuboramiz.

In [12]:
async def fix(section: dict, verdict: Verdict) -> tuple[Question | None, Verdict | None]:
    new_quiz = await generate_quiz(section, n=1, feedback=verdict.reason)
    new_q = new_quiz.questions[0]
    new_v = await evaluate(section, new_q)
    return (new_q, new_v) if new_v.ok else (None, new_v)

fixed = await asyncio.gather(*[fix(s, v) for s, q, v in bad])

for (s, q, v), (new_q, new_v) in zip(bad, fixed):
    status = "✓ tuzatildi" if new_q else "✗ tashlandi"
    print(f"{status} [{s['title']}]", new_q.question if new_q else new_v.reason)

## 9. Hammasi bitta funksiyada

Yuqoridagi qadamlarni bitta `build_quiz_bank()` ga yig'amiz. Mana shu — **workflow**: har safar bir xil 4 qadam, model qadamlarni tanlamaydi.

In [13]:
async def build_quiz_bank(sections: list[dict], n: int = 2) -> list[Question]:
    t0 = time.perf_counter()
    quizzes = await asyncio.gather(*[generate_quiz(s, n) for s in sections])           # 1. parallel generatsiya
    pairs = [(s, q) for s, quiz in zip(sections, quizzes) for q in quiz.questions]
    verdicts = await asyncio.gather(*[evaluate(s, q) for s, q in pairs])               # 2. parallel tekshirish
    good = [q for (s, q), v in zip(pairs, verdicts) if v.ok]
    bad = [(s, v) for (s, q), v in zip(pairs, verdicts) if not v.ok]
    fixed = await asyncio.gather(*[fix(s, v) for s, v in bad])                         # 3. optimizer (1 marta)
    good += [q for q, _ in fixed if q]
    print(f"{len(pairs)} test tuzildi | {len(bad)} yomon | {sum(1 for q, _ in fixed if q)} tuzatildi | "
          f"yakuniy {len(good)} | {time.perf_counter() - t0:.1f} s")
    return good

bank = await build_quiz_bank(SECTIONS)

10 test tuzildi | 0 yomon | 0 tuzatildi | yakuniy 10 | 8.5 s


## Xulosa

| Pattern | Bu notebookda | Qaysi muammoni hal qiladi |
|---|---|---|
| **Parallelization** | `asyncio.gather` — 5 bo'lim bir vaqtda | Tezlik: bo'limlar bir-biriga bog'liq emas |
| **Evaluator-optimizer** | `evaluate()` + `fix()` | Sifat: generator o'z xatosini ko'rmaydi |
| (bonus) **Kod + LLM tekshiruv** | `code_check()` + LLM | Qat'iy narsa kodda — tekin va aniq |

**Muhim joylar:**
- `Semaphore` — parallel ishlashda rate limit'dan saqlanish.
- Evaluator'ni **ataylab yomon test** bilan tekshirish — u ishlayotganiga ishonch hosil qilish.
- Optimizer **cheklangan** (1 marta) — cheksiz siklsiz.
- Agent yo'q: model hech qachon "keyingi qadam nima?" deb hal qilmaydi.

**Endi A3-topshiriq:** xuddi shu sxemani o'zingiz, **boshqa matn** bilan yozing (masalan, demo notebookdagi yo'riqnoma bandlari yoki o'zingiz tanlagan matn). Kodni ko'chirmang — shu notebookni yopib, sxemaga qarab yozing.